# Setup

In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import random
from dataclasses import dataclass
from collections import deque
import torch
import torch.nn as nn
import torch.optim as optim
from typing import Tuple, Optional, List

In [2]:
# Parameter/Variable/Defaults Selection
ENV_CONFIG = {"initialCash": 100000.0,    "maxInventory": 500,    "tickSize": 0.01,    "episodeLen": 1000,    "feePerShare": 0.001,    "seed": 123,
              "baseProb": 0.12,    "maxProb": 0.60,    "lamScale": 100.0,    "imbFillWeight": 1.2,    "intensityFillWeight": 1.0,    "tickDecay": 0.4,           
              "aggressiveBonus": 3.5,    "toxImb": 0.3,    "toxPxChg": 0.4,    "toxIntensity": 0.2,    "toxSpread": 0.1,    "spreadToxThreshold": 0.15,
              "fillFracMin": 0.25,    "fillFracMax": 0.75,    "idlePenalty": 1e-5}

AGENT_CONFIG = {"stateSize": 24,    "actionSize": 20,    "learningRate": 1e-3,    "gamma": 0.95,    "epsilonStart": 1.0,    "epsilonEnd": 0.05,
                "epsilonDecay": 0.994,    "batchSize": 256,    "targetUpdateFreq": 2000,    "bufferCapacity": 200000,    "lstmLr": 5e-4,
                "lambdaMax": (0.5, 0.3, 1.0),    "defaultLambdas": (0.01, 0.05, 0.75),    "regimeWindow": 100,    "lambdaRefreshFreq": 50}

FEATURE_CONFIG = {"VolatilityWindow": (500,),    "downsampling": 100,    "hawkMu": 1.0,    
                  "hawkAlpha": 0.5,          "hawkBeta": 2.0,      "historyMaxLen": 2000}

AS_CONFIG = {"gamma":      0.1,    "kappa":      1.5,   "episodeLen": 1000,   "feePerShare": 0.001}

# Data Classes

In [3]:
@dataclass(frozen = True)
class LOBSnapshot:
    timestamp: int
    bid_prices: np.ndarray
    ask_prices: np.ndarray
    bid_sizes: np.ndarray
    ask_sizes: np.ndarray
    bid_counts: np.ndarray
    ask_counts: np.ndarray
    action: str
    side: str

## Creating Microstructure Features

In [4]:
class FeatureEngine:
    def __init__(self, VolatilityWindow=(500,), downsampling=100, hawkMu=1.0, hawkAlpha=0.5, hawkBeta=2.0, historyMaxLen=2000):
        self.VolatilityWindow, self.volAnnualized = list(VolatilityWindow), 1
        self.downsampling = downsampling
        self.hawkMu, self.hawkAlpha, self.hawkBeta, self.hawkLambda, self.hawkLastPeriod = float(hawkMu), float(hawkAlpha), float(hawkBeta), float(hawkMu/downsampling), None
        self.historyWindow, self.priceHistory = deque(maxlen=historyMaxLen), deque(maxlen=max(VolatilityWindow))
        self.lastMidprice = None

    def createFeatures(self, lob: LOBSnapshot):
        bestBid, bidSize = float(lob.bid_prices[0]), float(lob.bid_sizes[0])
        bestAsk, askSize = float(lob.ask_prices[0]), float(lob.ask_sizes[0])
        minLevelSearch = min(5, len(lob.bid_prices), len(lob.ask_prices), len(lob.bid_sizes), len(lob.ask_sizes))
        
        # Kick out invalid periods
        if not (np.isfinite(bestBid) and np.isfinite(bestAsk)) or bestBid <= 0 or bestAsk <= 0:
            return None
        
        # Set equal weight midprice
        midprice = 0.5 * (bestBid + bestAsk)
        totalVolume = bidSize + askSize
        
        # Check if crossing spread:
        if bestAsk <= bestBid:
            spread = 0.0
        else:
            spread = bestAsk - bestBid
        
        # Finding microprice and pressures on sides of LOB
        if totalVolume > 0:
            microprice = (bestBid * askSize + bestAsk * bidSize) / totalVolume
            imbalance = (bidSize - askSize) / totalVolume
        else:
            microprice = midprice
            imbalance = 0.0
        
        # Identify VWAP
        bidPriceXSize = askPriceXSize = bidDepth = askDepth = 0.0
        
        for i in range(minLevelSearch):
            currBidP, currAskP = float(lob.bid_prices[i]), float(lob.ask_prices[i])
            currBidS, currAskS = float(lob.bid_sizes[i]), float(lob.ask_sizes[i])
            
            if currBidP > 0 and currBidS > 0:
                bidPriceXSize += currBidP * currBidS
                bidDepth += currBidS
            
            if currAskP > 0 and currAskS > 0:
                askPriceXSize += currAskP * currAskS
                askDepth += currAskS
        
        if bidDepth > 0:
            bidVWAP = (bidPriceXSize / bidDepth)
        else:
            bidVWAP = float(lob.bid_prices[0])
        
        if askDepth > 0:
            askVWAP = (askPriceXSize / askDepth)
        else:
            askVWAP = float(lob.ask_prices[0])
        
        totalDepth = bidDepth + askDepth
        if totalDepth > 0:
            depthImbal = (bidDepth - askDepth) / totalDepth
        else:
            depthImbal = 0.0
        
        # LOB price impact proxy via slope over levels
        bidSlopeSum = askSlopeSum = bidSlopeCount = askSlopeCount = 0
        
        for i in range(minLevelSearch - 1):
            bidPrice0 = float(lob.bid_prices[i])
            bidPrice1 = float(lob.bid_prices[i+1])
            bidSize0 = float(lob.bid_sizes[i])
            
            askPrice0 = float(lob.ask_prices[i])
            askPrice1 = float(lob.ask_prices[i+1])
            askSize0 = float(lob.ask_sizes[i])
            
            if bidPrice0 > 0 and bidPrice1 > 0 and bidSize0 > 0:
                bidSlopeSum += abs(bidPrice1 - bidPrice0)
                bidSlopeCount += 1
            
            if askPrice0 > 0 and askPrice1 > 0 and askSize0 > 0:
                askSlopeSum += abs(askPrice1 - askPrice0)
                askSlopeCount += 1
        
        bidSlope = float(bidSlopeSum / bidSlopeCount) if bidSlopeCount > 0 else 0.0
        askSlope = float(askSlopeSum / askSlopeCount) if askSlopeCount > 0 else 0.0
        
        # Add midprice to the volatility index over time
        self.priceHistory.append(midprice)
        if len(self.priceHistory) < 10:
            vol500 = 0.0
        else:
            prices = np.asarray(self.priceHistory, dtype=np.float64)
            rets = np.diff(np.log(prices + 1e-12))
            vol500 = float(np.std(rets[-500:])) if len(rets) >= 500 else float(np.std(rets))
        
        # Hawkes intensity
        aggressiveAction = lob.action in ("A", "T")
        
        if self.hawkLastPeriod is None:
            self.hawkLastPeriod = int(lob.timestamp)
            self.hawkLambda = self.hawkMu + (self.hawkAlpha if aggressiveAction else 0.0)
            dt = 0.0
        else:
            dt = max(0.0, (int(lob.timestamp) - self.hawkLastPeriod)/1e9)
            decay = np.exp(-self.hawkBeta * dt)
            self.hawkLambda = self.hawkMu + (self.hawkLambda - self.hawkMu) * decay
            if aggressiveAction:
                self.hawkLambda += self.hawkAlpha
            self.hawkLastPeriod = int(lob.timestamp)
        
        hawkLambda = self.hawkLambda
        excess = max(0.0, self.hawkLambda - self.hawkMu)
        hawkIntegrated = self.hawkMu * dt + (excess / self.hawkBeta) * (1.0 - np.exp(-self.hawkBeta * dt)) if dt > 0 else 0.0
        
        # Update and compute order flow
        self.historyWindow.append((int(lob.timestamp), str(lob.action), str(lob.side)))
        
        windowNS = 5_000_000_000 * self.downsampling
        windowSec = 5.0 * self.downsampling
        
        cutoff = int(lob.timestamp - windowNS)
        adds = cancels = total = 0
        
        for ts, action, _ in self.historyWindow:
            if ts >= cutoff:
                total += 1
                if action == "A":
                    adds += 1
                elif action == "C":
                    cancels += 1
        
        addRate = adds / windowSec
        cancelRate = cancels / windowSec
        cancelAddRatio = float(cancels / (adds + 1e-6))
        orderIntensity = float(total / windowSec)
        
        # Price change
        if self.lastMidprice is not None and self.lastMidprice > 0:
            dpx = midprice - self.lastMidprice
            dpxBPS = (dpx / self.lastMidprice) * 1e4
        else:
            dpx = dpxBPS = 0.0
        
        self.lastMidprice = midprice
        
        spreadBPS = (spread / midprice) * 10000.0
        
        return {"timestamp": int(lob.timestamp), 
                "mid_price": float(midprice), "microprice": float(microprice), 
                "best_bid": float(bestBid), "best_ask": float(bestAsk), "spread": float(spread), "spread_bps": float(spreadBPS), 
                "imbalance_l1": float(imbalance), "bid_vwap_5": float(bidVWAP), "ask_vwap_5": float(askVWAP), 
                "bid_depth_5": float(bidDepth), "ask_depth_5": float(askDepth), "depth_imbalance_5": float(depthImbal),
                "bid_slope": float(bidSlope), "ask_slope": float(askSlope), "vol_500tick": float(vol500), 
                "hawkes_lambda_market": float(hawkLambda), "hawkes_integrated_market": float(hawkIntegrated), 
                "add_rate": float(addRate), "cancel_rate": float(cancelRate), "cancel_add_ratio": float(cancelAddRatio), 
                "order_intensity": float(orderIntensity), "price_change": float(dpx), "price_change_bps": float(dpxBPS),}

# Reinforcement/Deep Learning Framework Classes

In [5]:
class DQNetwork(nn.Module):
    def __init__(self, stateSize: int, actionSize: int, hiddenSizes=(256, 256, 128)):
        super().__init__()
        layers = []
        inDim = stateSize
        for h in hiddenSizes:
            layers.append(nn.Linear(inDim, h))
            layers.append(nn.ReLU())
            inDim = h
        layers.append(nn.Linear(inDim, actionSize))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

In [6]:
class LSTMMetaLearner(nn.Module):
    def __init__(self, inputSize=10, hiddenSize=128, outputSize=3, lambdaMax=(0.5, 0.5, 0.5)):
        super().__init__()
        self.lstm = nn.LSTM(inputSize, hiddenSize, batch_first=True, num_layers=2)
        self.fc = nn.Sequential(
            nn.Linear(hiddenSize, 32),
            nn.ReLU(),
            nn.Linear(32, outputSize),
        )
        self.register_buffer("lambdaMax", torch.tensor(lambdaMax, dtype=torch.float32))
    
    def forward(self, x):
        _, (hN, _) = self.lstm(x)
        raw = self.fc(hN[-1])
        lambdas = torch.sigmoid(raw) * self.lambdaMax
        return lambdas

In [7]:
class MetaDQNAgent:
    def __init__(self, stateSize: int, actionSize: int, learningRate: float = 1e-3, gamma: float = 0.99,
                 epsilonStart: float = 1.0, epsilonEnd: float = 0.05, epsilonDecay: float = 0.9995,
                 batchSize: int = 256, targetUpdateFreq: int = 2000, bufferCapacity: int = 200_000,
                 lstmLr: float = 1e-4, lambdaMax: Tuple[float, float, float] = (0.5, 0.5, 0.5),
                 defaultLambdas: Tuple[float, float, float] = (0.01, 0.05, 0.02), regimeWindow: int = 60,
                 lambdaRefreshFreq: int = 30, maxInventory: int = 500):
        # Set up data/info & State
        self.stateSize = stateSize
        self.actionSize = actionSize
        self.gamma = gamma
        self.batchSize = batchSize
        self.targetUpdateFreq = targetUpdateFreq
        self.maxInventory = maxInventory

        # Epsilon decay definition
        self.epsilon = epsilonStart
        self.epsilonEnd = epsilonEnd
        self.epsilonDecay = epsilonDecay

        # Torch using GPU on home device, optional CPU run
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # First DQN
        self.qNet = DQNetwork(stateSize, actionSize).to(self.device)
        # Second/Target DQN
        self.targetNet = DQNetwork(stateSize, actionSize).to(self.device)
        self.targetNet.load_state_dict(self.qNet.state_dict())
        self.targetNet.eval()

        # LSTM Learning
        self.metaLearner = LSTMMetaLearner(inputSize=17, hiddenSize=64, outputSize=3, lambdaMax=lambdaMax).to(self.device)

        # Gradient Optimization + Learning
        self.dqnOptimizer = optim.Adam(self.qNet.parameters(), lr=learningRate)
        self.metaOptimizer = optim.Adam(self.metaLearner.parameters(), lr=lstmLr, weight_decay = 1e-4)

        # Loss Function
        self.lossFn = nn.SmoothL1Loss()
        
        self.replayBuffer = deque(maxlen=bufferCapacity)

        self.trainSteps = 0
        # deque capped at regimeWindow — we only ever need the last N states
        self.regimeWindow = regimeWindow
        self.stateHistory = deque(maxlen=regimeWindow)

        # Initial Values + param refresh
        self.defaultLambdas = np.asarray(defaultLambdas, dtype=np.float32)
        self.currentLambdas = self.defaultLambdas.copy()
        self.lambdaRefreshFreq = lambdaRefreshFreq

    def selectAction(self, state: np.ndarray, training: bool = True) -> int:
        # Sets exploration after action
        if training and random.random() < self.epsilon:
            return random.randrange(self.actionSize)

        with torch.no_grad():
            s = torch.as_tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
            return int(torch.argmax(self.qNet(s), dim=1).item())

    def storeTransition(self, state, action, reward, nextState, done):
        # puts together info inro the buffer
        s = np.asarray(state, dtype=np.float32)
        ns = np.asarray(nextState, dtype=np.float32) if nextState is not None else s
        self.replayBuffer.append((s, int(action), float(reward), ns, float(done)))

    def trainStep(self) -> Optional[float]:
        # Updates the DQN Policy update
        if len(self.replayBuffer) < self.batchSize:
            return None
        
        # Take random sample from regime snapshots
        batch = random.sample(self.replayBuffer, self.batchSize)
        states, actions, rewards, nextStates, dones = zip(*batch)

        # Set up info in the Q-Network
        states     = torch.as_tensor(np.stack(states),                       dtype=torch.float32, device=self.device)
        actions    = torch.as_tensor(np.asarray(actions,  dtype=np.int64),   dtype=torch.int64,   device=self.device)
        rewards    = torch.as_tensor(np.asarray(rewards,  dtype=np.float32), dtype=torch.float32, device=self.device)
        nextStates = torch.as_tensor(np.stack(nextStates),                   dtype=torch.float32, device=self.device)
        dones      = torch.as_tensor(np.asarray(dones,    dtype=np.float32), dtype=torch.float32, device=self.device)

        # Outputs
        qValues = self.qNet(states).gather(1, actions.view(-1, 1)).squeeze(1)
        
        with torch.no_grad():
            nextActions  = torch.argmax(self.qNet(nextStates), dim=1)
            nextQValues  = self.targetNet(nextStates).gather(1, nextActions.view(-1, 1)).squeeze(1)
            targets      = rewards + (1.0 - dones) * self.gamma * nextQValues

        loss = self.lossFn(qValues, targets)
        self.dqnOptimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.qNet.parameters(), 1.0)
        self.dqnOptimizer.step()

        self.trainSteps += 1
        if self.trainSteps % self.targetUpdateFreq == 0:
            self.targetNet.load_state_dict(self.qNet.state_dict())

        return float(loss.item())

    def extractRegimeFeatures(self) -> np.ndarray:
        if len(self.stateHistory) < self.regimeWindow:
            return np.zeros(17, dtype=np.float32)
    
        s = np.asarray(self.stateHistory, dtype=np.float32)
    
        midprices  = s[:, 0]
        returns    = np.diff(midprices) / midprices[:-1]
        inv_raw    = s[:, 23] * float(self.maxInventory)
        # Initial extracted features
        return np.array([
            # return volatility
            np.std(returns) * 10000,
            
            # mean abs ret
            np.mean(np.abs(returns)) * 10000,
            
            # mean price change bps
            np.mean(s[:, 20]),
            
            # mean spread bps
            np.mean(s[:, 3]),
            
            # STDev Spread
            np.std(s[:, 3]),
            
            # mean depth imbal at 5 level
            np.mean(s[:, 9]),
            
            # depth imbal stdev
            np.std(s[:, 9]),
            
            # mean bid slope
            np.mean(s[:, 10]),
            
            # mean ask_slope
            np.mean(s[:, 11]),
            
            # Mean hawked_lambda_market
            np.mean(s[:, 13]),
            
            # mean hawkes_integrated_market
            np.mean(s[:, 14]),
            
            # mean order intensity
            np.mean(s[:, 18]),
            
            # mean cancel_rate
            np.mean(s[:, 15]),
            
            # mean cancel_rate
            np.mean(s[:, 16]),
            
            # Mean Inventory
            np.mean(inv_raw),
            
            # STDev inventory
            np.std(inv_raw),                    
            
            # Max inventory exposure
            np.max(np.abs(inv_raw))], dtype=np.float32)

    def updateLambdas(self) -> np.ndarray:
        # Pass lambdas into the LSTM
        if len(self.stateHistory) < self.regimeWindow:
            return self.currentLambdas

        regime = self.extractRegimeFeatures()
        with torch.no_grad():
            x = torch.as_tensor(regime, dtype=torch.float32, device=self.device).view(1, 1, -1)
            self.currentLambdas = self.metaLearner(x).view(-1).cpu().numpy().astype(np.float32)

        return self.currentLambdas

    def trainEpisode(self, env, updateMeta: bool = False) -> Tuple[dict, float]:
        # Train on our current time series
        state = env.reset(episodeLen=1000)
        self.stateHistory.clear()
        self.currentLambdas = self.defaultLambdas.copy()

        wealthAtLambdaStart = env.cash + env.inventory * state[0]
        lambdaPerformance = []

        totalReward = 0.0
        step = 0

        while True:
            if (step % self.lambdaRefreshFreq == 0) and (len(self.stateHistory) >= self.regimeWindow):
                regime = self.extractRegimeFeatures()
                oldLambdas = self.currentLambdas.copy()
                self.updateLambdas()
                currentWealth = env.cash + env.inventory * state[0]
                lambdaPerformance.append((regime, oldLambdas, currentWealth - wealthAtLambdaStart))
                wealthAtLambdaStart = currentWealth

            action = self.selectAction(state, training=True)
            nextState, reward, done, _ = env.step(action, lambdaParams=tuple(self.currentLambdas))

            self.storeTransition(state, action, reward, nextState, done)

            if step % 4 == 0:
                self.trainStep()

            self.stateHistory.append(state)
            totalReward += float(reward)
            state = nextState if nextState is not None else state
            step += 1

            if done:
                break

        # Decay epsilon
        self.epsilon = max(self.epsilon * self.epsilonDecay, self.epsilonEnd)

        # Train meta learner
        if updateMeta and len(lambdaPerformance) >= 5:
            pnls = np.array([pnl for _, _, pnl in lambdaPerformance], dtype=np.float32)
            threshold = np.quantile(pnls, 0.5)
            goodConfigs = [(regime, lambdas) for regime, lambdas, pnl in lambdaPerformance if pnl >= threshold]

            if goodConfigs:
                regimes = torch.as_tensor(
                    np.stack([r for r, _ in goodConfigs]),
                    dtype=torch.float32, device=self.device
                ).view(len(goodConfigs), 1, -1)
                targets = torch.as_tensor(
                    np.stack([lam for _, lam in goodConfigs]),
                    dtype=torch.float32, device=self.device
                )
                loss = nn.MSELoss()(self.metaLearner(regimes), targets)
                self.metaOptimizer.zero_grad()
                loss.backward()
                self.metaOptimizer.step()

        return env.getMetrics(), totalReward

    def evaluateEpisode(self, env) -> Tuple[dict, float]:
        # Measuring performance on time series, so no taining occurs
        state = env.reset()
        self.stateHistory.clear()
        self.currentLambdas = self.defaultLambdas.copy()

        totalReward = 0.0
        step = 0

        while True:
            if (step % self.lambdaRefreshFreq == 0) and (len(self.stateHistory) >= self.regimeWindow):
                self.updateLambdas()

            action = self.selectAction(state, training=False)
            nextState, reward, done, _ = env.step(action, lambdaParams=tuple(self.currentLambdas))

            self.stateHistory.append(state)
            totalReward += float(reward)
            state = nextState if nextState is not None else state
            step += 1

            if done:
                break

        return env.getMetrics(), totalReward

# AMM Engine & Environment Sim

In [8]:
class AMMEnvironment:

    COLS = [
        "mid_price",
        "microprice", "spread", "spread_bps", "imbalance_l1",
        "bid_vwap_5", "ask_vwap_5", "bid_depth_5", "ask_depth_5",
        "depth_imbalance_5", "bid_slope", "ask_slope",
        "vol_500tick",
        "hawkes_lambda_market", "hawkes_integrated_market",
        "add_rate", "cancel_rate", "cancel_add_ratio", "order_intensity",
        "price_change", "price_change_bps",
        "best_bid", "best_ask"
    ]

    def __init__(self, featureData: pd.DataFrame,
                 initialCash: float = 100_000.0, maxInventory: int = 500, tickSize: float = 0.01,
                 episodeLen: int | None = None, feePerShare: float = 0.001, seed: int = 123, baseProb: float = 0.002, maxProb: float = 0.05,
                 lamScale: float = 100.0, imbFillWeight: float = 0.8, intensityFillWeight: float = 0.7, tickDecay: float = 0.9,
                 aggressiveBonus: float = 1.5, toxImb: float = 0.3, toxPxChg: float = 0.4, toxIntensity: float = 0.2, toxSpread: float = 0.1,
                 spreadToxThreshold: float = 0.15, fillFracMin: float = 0.25, fillFracMax: float = 0.75, idlePenalty: float = 1e-5):
        # Take in info
        self.initialCash = float(initialCash)
        self.maxInventory = int(maxInventory)
        self.tickSize = float(tickSize)
        self.episodeLen = episodeLen
        self.feePerShare = float(feePerShare)
        self.rng = np.random.default_rng(seed)
        self.baseProb = float(baseProb)
        self.maxProb = float(maxProb)
        self.lamScale = float(lamScale)
        self.imbFillWeight = float(imbFillWeight)
        self.intensityFillWeight = float(intensityFillWeight)
        self.tickDecay = float(tickDecay)
        self.aggressiveBonus = float(aggressiveBonus)
        self.toxImb = float(toxImb)
        self.toxPxChg = float(toxPxChg)
        self.toxIntensity = float(toxIntensity)
        self.toxSpread = float(toxSpread)
        self.spreadToxThreshold = float(spreadToxThreshold)
        self.fillFracMin = float(fillFracMin)
        self.fillFracMax = float(fillFracMax)
        self.idlePenalty = float(idlePenalty)

        missing = [c for c in self.COLS if c not in featureData.columns]
        if missing:
            raise ValueError(f"Missing required columns: {missing}")

        self.X = featureData[self.COLS].to_numpy(dtype=np.float32)
        self.N = int(self.X.shape[0])
        if self.N <= 1:
            raise ValueError("featureData too small for training.")

        self.stateSize = 24
        self.actionSpaceSize = 20

        self.reset()

    def decodeAction(self, actionId: int):
        # Intakes an actiona and sets a bid-ask with size as well, with scaling
        if self.inventory <= -0.95 * self.maxInventory:
            actionId = min(actionId, 3)
        if self.inventory >= 0.95 * self.maxInventory:
            actionId = 4

        r = self.X[self.idx]
        bestBid = float(r[21])
        bestAsk = float(r[22])
        sizes = [100, 200, 500]

        inv = float(self.inventory)
        maxInv = float(self.maxInventory)
        invRatio = inv / maxInv

        damp = max(0.10, 1.0 - abs(invRatio))
        bidSkew = max(0.05, 1.0 - 2.0 * invRatio)
        askSkew = max(0.05, 1.0 + 2.0 * invRatio)

        def sized(baseSize: int, side: str) -> int:
            
            if side == "bid":
                s = int(round(baseSize * damp * bidSkew))
                cap = int(self.maxInventory - self.inventory)
            else:
                s = int(round(baseSize * damp * askSkew))
                cap = int(self.maxInventory + self.inventory)
            s = min(s, 200)
            return max(0, min(s, cap))

        if actionId == 0:
            return None, None
        if actionId == 19:
            return None, None

        # Bid at best bid
        if 1 <= actionId <= 3:
            base = sizes[actionId - 1]
            size = sized(base, "bid")
            return ((bestBid, size), None) if size > 0 else (None, None)

        # Ask at best ask
        if 4 <= actionId <= 6:
            base = sizes[actionId - 4]
            size = sized(base, "ask")
            return (None, (bestAsk, size)) if size > 0 else (None, None)

        # Both sides at best
        if 7 <= actionId <= 9:
            base = sizes[actionId - 7]
            bidSize = sized(base, "bid")
            askSize = sized(base, "ask")
            bidQ = (bestBid, bidSize) if bidSize > 0 else None
            askQ = (bestAsk, askSize) if askSize > 0 else None
            return bidQ, askQ

        # Bid 1 tick back
        if 10 <= actionId <= 12:
            base = sizes[actionId - 10]
            size = sized(base, "bid")
            px = bestBid - self.tickSize
            return ((px, size), None) if size > 0 else (None, None)

        # Ask 1 tick back
        if 13 <= actionId <= 15:
            base = sizes[actionId - 13]
            size = sized(base, "ask")
            px = bestAsk + self.tickSize
            return (None, (px, size)) if size > 0 else (None, None)

        # Both sides 1 tick back
        if 16 <= actionId <= 18:
            base = sizes[actionId - 16]
            bidSize = sized(base, "bid")
            askSize = sized(base, "ask")
            bidQ = (bestBid - self.tickSize, bidSize) if bidSize > 0 else None
            askQ = (bestAsk + self.tickSize, askSize) if askSize > 0 else None
            return bidQ, askQ

        return None, None

    def simulateFills(self, bidQuote, askQuote):
        # assigns fill prob or nor relative to activity, updating inventory/cash
        r = self.X[self.idx]
        imb = float(r[4])
        lam = float(r[13])
        bestBid = float(r[21])
        bestAsk = float(r[22])
        intensity = min(1.0, lam / self.lamScale)

        fills = { "bid_filled": False, "ask_filled": False, "bid_fill_price": 0.0, "ask_fill_price": 0.0,
                  "bid_fill_size": 0, "ask_fill_size": 0, "micro_t": float(r[1]), "bestBid_t": bestBid, 
                  "bestAsk_t": bestAsk}
        # Simulates the probability of getting order filled against the market
        def fillProb(price, side):
            if side == "bid":
                dTicks = (bestBid - price) / self.tickSize
                imbEffect = 1.0 + self.imbFillWeight * imb
            else:
                dTicks = (price - bestAsk) / self.tickSize
                imbEffect = 1.0 - self.imbFillWeight * imb
            intensityEffect = 1.0 + self.intensityFillWeight * intensity
            competitiveness = np.exp(-self.tickDecay * max(dTicks, 0.0)) * (self.aggressiveBonus if dTicks < 0 else 1.0)
            return float(np.clip(self.baseProb * intensityEffect * imbEffect * competitiveness, 0.0, self.maxProb))

        #if we have a quote, then check and update
        if bidQuote is not None:
            price, size = bidQuote
            size = int(min(size, self.maxInventory - self.inventory))
            if size > 0 and self.rng.random() < fillProb(price, "bid"):
                filledSize = max(1, int(size * (self.fillFracMin + self.fillFracMax * self.rng.random())))
                fills["bid_filled"] = True
                fills["bid_fill_price"] = float(price)
                fills["bid_fill_size"] = filledSize
                self.inventory += filledSize
                self.cash -= float(price) * float(filledSize)
                self.totalTrades += 1
        if askQuote is not None:
            price, size = askQuote
            size = int(min(size, self.maxInventory + self.inventory))
            if size > 0 and self.rng.random() < fillProb(price, "ask"):
                filledSize = max(1, int(size * (self.fillFracMin + self.fillFracMax * self.rng.random())))
                fills["ask_filled"] = True
                fills["ask_fill_price"] = float(price)
                fills["ask_fill_size"] = filledSize
                self.inventory -= filledSize
                self.cash += float(price) * float(filledSize)
                self.totalTrades += 1

        return fills

    def step(self, action: int, lambdaParams=(0.01, 0.05, 0.02), debugReward: bool = False):
        # Main looping, reading in info at t, calling the other functions &
        # calculating PnL, fees, etc. Also updates rewards
        rT = self.X[self.idx]
        midT = float(rT[0])
        bid, ask = self.decodeAction(int(action))
        self.bidPosted, self.askPosted = bid, ask
        fills = self.simulateFills(bid, ask)

        # Accumulating PnL vals
        realizedPnl = 0.0
        if fills["bid_filled"]:
            realizedPnl += (midT - fills["bid_fill_price"]) * fills["bid_fill_size"]
        if fills["ask_filled"]:
            realizedPnl += (fills["ask_fill_price"] - midT) * fills["ask_fill_size"]

        # Fees are determined by fees Per Share (which we set at midprice) * size of fill
        fees = self.feePerShare * (float(fills.get("bid_fill_size", 0)) +float(fills.get("ask_fill_size", 0)))

        # t+1
        self.idx += 1
        self.stepsInEp += 1
        # Check if step done by index/ep length
        done = (self.idx >= self.N)
        if self.episodeLen is not None:
            done = done or (self.stepsInEp >= self.episodeLen)
        if done:
            midTp1 = midT
            nextState = None
        else:
            rTp1 = self.X[self.idx]
            midTp1 = float(rTp1[0])
            invNorm = np.float32(self.inventory / self.maxInventory)
            # Tracks all the info in our vector
            nextState = np.array([rTp1[0], rTp1[1], rTp1[2], rTp1[3], rTp1[4], rTp1[5], rTp1[6], rTp1[7], rTp1[8], rTp1[9], rTp1[10], rTp1[11], rTp1[12], 
                                  rTp1[13], rTp1[14], rTp1[15], rTp1[16], rTp1[17], rTp1[18], rTp1[19], rTp1[20], rTp1[21], rTp1[22], invNorm], dtype=np.float32)

        # Wealth as pos value in the reward
        wealthTp1 = self.cash + self.inventory * midTp1
        
        # Change in wealth
        dW = wealthTp1 - self.prevWealth
        
        # Update wealth
        self.prevWealth = wealthTp1
        
        # Import default lambdas
        lam0, lam1, lam2 = lambdaParams
        # Normalize inventory by arbitrary max values
        invRatio = self.inventory / self.maxInventory

        # Lin + Quadratic penalty
        invPenalty = lam0 * abs(invRatio) + lam1 * (invRatio ** 2)

        # Toxicity
        toxicity  = (self.toxImb * abs(float(rT[4])) + self.toxIntensity * min(1.0, float(rT[13]) / self.lamScale) + self.toxPxChg * abs(float(rT[20])))
        toxPenalty = lam2 * toxicity

        # Total Reward at step t
        reward = float(dW - invPenalty - toxPenalty - fees)

        # Record Everything
        self.rewardHistory.append(reward)
        self.wealthHistory.append(float(wealthTp1))
        self.invHistory.append(int(self.inventory))

        # Created this to track movements in steps
        info = {"fills": fills, "inventory": self.inventory, "cash": self.cash, "pnl": wealthTp1 - self.initialCash,  "totalTrades": self.totalTrades}
        return nextState, reward, done, info

    def reset(self, startIdx: int | None = None, episodeLen: int | None = None):
        # Resets the environment after each time series we run
        if episodeLen is not None:
            self.episodeLen = int(episodeLen)
            startIdx = int(self.rng.integers(0, max(1, self.N - self.episodeLen - 1)))
        self.cash = self.initialCash
        self.inventory = 0
        self.totalTrades = 0
        self.bidPosted = None
        self.askPosted = None
        self.start = 0 if startIdx is None else int(np.clip(startIdx, 0, self.N - 1))
        self.idx = self.start
        self.stepsInEp = 0
        self.wealthHistory = []
        self.invHistory = []
        self.rewardHistory = []
        self.prevWealth = self.cash + self.inventory * float(self.X[self.idx][0])
        self.wealthHistory.append(float(self.prevWealth))
        self.invHistory.append(int(self.inventory))

        r0 = self.X[self.idx]
        invNorm = np.float32(self.inventory / self.maxInventory)
        return np.array([r0[0], r0[1], r0[2], r0[3], r0[4], r0[5], r0[6], r0[7],
                         r0[8], r0[9], r0[10], r0[11], r0[12], r0[13], r0[14], r0[15],
                         r0[16], r0[17], r0[18], r0[19], r0[20], r0[21], r0[22], invNorm], dtype=np.float32)

    def getMetrics(self):
        # Basic summary stats
        mid = float(self.X[max(self.idx - 1, 0), 0])
        wealth = self.cash + self.inventory * mid
        pnl = wealth - self.initialCash

        if len(self.rewardHistory) > 1:
            rr = np.asarray(self.rewardHistory, dtype=np.float32)
            sharpe = float(np.mean(rr) / (np.std(rr) + 1e-6) * np.sqrt(len(rr)))
        else:
            sharpe = 0.0

        return {"finalPnl": float(pnl), "finalWealth": float(wealth), "finalInventory": int(self.inventory), "totalTrades": int(self.totalTrades), "sharpeRatio": float(sharpe), "steps": int(self.stepsInEp)}

# Baseline

In [9]:
class AvellanedaStoikov:

    def __init__(self,
                 gamma: float = 0.1,
                 kappa: float = 1.5,
                 episodeLen: int = 1000,
                 feePerShare: float = 0.001):
        # Risk aversion
        self.gamma     = gamma
        # market depth
        self.kappa     = kappa        
        self.episodeLen = episodeLen
        self.feePerShare = feePerShare

    def computeQuotes(self, mid: float, vol: float, inventory: int, maxInventory: int, timeRemaining: float):
        # Make info for reward
        invRatio = inventory / maxInventory
        reservationPrice = mid - invRatio * self.gamma * (vol ** 2) * timeRemaining
        optimalSpread    = (self.gamma * (vol ** 2) * timeRemaining + (2.0 / self.gamma) * np.log(1.0 + self.gamma / self.kappa))
        halfSpread = optimalSpread / 2.0
        # Set price
        bidPrice   = reservationPrice - halfSpread
        askPrice   = reservationPrice + halfSpread

        return bidPrice, askPrice

    def evaluateEpisode(self, env) -> tuple:
        # Resets ep and values
        state     = env.reset()
        totalReward = 0.0
        step      = 0

        # While in period do this, eventually will stop
        while True:
            mid       = float(state[0])
            vol       = float(state[12])
            bestBid   = float(state[21])
            bestAsk   = float(state[22])
            timeRemaining = max(0.01, 1.0 - step / self.episodeLen)
            bidPx, askPx = self.computeQuotes(mid, vol, env.inventory, env.maxInventory, timeRemaining)
            # Clip to best bid/ask so quotes are always reasonable
            bidPx = min(bidPx, bestBid)
            askPx = max(askPx, bestAsk)
            bidQuote = (bidPx, 100)
            askQuote = (askPx, 100)

            # Try to fill it
            fills = env.simulateFills(bidQuote, askQuote)

            # Step environment manually
            env.idx        += 1
            env.stepsInEp  += 1

            done = (env.idx >= env.N)
            if env.episodeLen is not None:
                done = done or (env.stepsInEp >= env.episodeLen)

            if done:
                midTp1    = mid
                nextState = None
            else:
                rTp1      = env.X[env.idx]
                midTp1    = float(rTp1[0])
                invNorm   = np.float32(env.inventory / env.maxInventory)
                nextState = np.array([rTp1[0], rTp1[1], rTp1[2], rTp1[3], rTp1[4], rTp1[5], rTp1[6], rTp1[7],
                                      rTp1[8], rTp1[9], rTp1[10], rTp1[11], rTp1[12], rTp1[13], rTp1[14], rTp1[15],
                                      rTp1[16], rTp1[17], rTp1[18], rTp1[19], rTp1[20], rTp1[21], rTp1[22], invNorm], dtype=np.float32)
            # Update wealth, change in wealth, update fees for fills and create reward
            wealthTp1      = env.cash + env.inventory * midTp1
            dW             = wealthTp1 - env.prevWealth
            env.prevWealth = wealthTp1
            fees = self.feePerShare * (float(fills.get("bid_fill_size", 0)) + float(fills.get("ask_fill_size", 0)))
            reward = float(dW - fees)
            # Record Everything
            env.rewardHistory.append(reward)
            env.wealthHistory.append(float(wealthTp1))
            env.invHistory.append(int(env.inventory))
            totalReward += reward
            state = nextState if nextState is not None else state
            step += 1
            # Stop loop if we hit end of ep
            if done:
                break
        return env.getMetrics(), totalReward